In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [6]:
text = "Hello world"
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(vocab_size)

stoi = {ch: i for i, ch in enumerate(chars)}  # string to integer
itos = {i: ch for i, ch in enumerate(chars)}  # integer to string
encode = lambda s: [stoi[c] for c in s]  # encode a string to a list of integers
decode = lambda l: ''.join([itos[i] for i in l])  # decode a list of integers to a string

8


In [3]:
block_size = 4
X = []
Y = []

for i in range(len(text) - block_size):
    chunk = text[i:i + block_size]
    target = text[i+1:i + block_size + 1]
    X.append(encode(chunk))
    Y.append(encode(target))

In [4]:
X

[[1, 3, 4, 4],
 [3, 4, 4, 5],
 [4, 4, 5, 0],
 [4, 5, 0, 7],
 [5, 0, 7, 5],
 [0, 7, 5, 6],
 [7, 5, 6, 4]]

In [5]:
Y

[[3, 4, 4, 5],
 [4, 4, 5, 0],
 [4, 5, 0, 7],
 [5, 0, 7, 5],
 [0, 7, 5, 6],
 [7, 5, 6, 4],
 [5, 6, 4, 2]]

In [11]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, n_embed=10):
        super(TinyGPT, self).__init__()
        self.embedding = nn.Embedding(vocab_size, n_embed)
        self.linear = nn.Linear(n_embed, vocab_size)
    def forward(self, x):
        embeds = self.embedding(x)  # (N, block_size, n_embed)
        logits = self.linear(embeds)
        return logits

In [14]:
model = TinyGPT(vocab_size)
logits = model(torch.tensor(X, dtype=torch.long))  # (N, block_size, vocab_size)
print(logits.shape)

torch.Size([7, 4, 8])


In [16]:
Y = torch.tensor(Y, dtype=torch.long)  # (N, block_size)
loss_fn = nn.CrossEntropyLoss()
logits = logits.view(-1, vocab_size)
print(logits.shape)
targets = Y.view(-1)
print(targets.shape)

loss = loss_fn(logits, targets)
print(loss.item())  # Print the loss value

torch.Size([28, 8])
torch.Size([28])
2.2269957065582275


> The obtained loss values is 2.22 which is good but it has some problems because the model is not using uniform distribution for prediction.

```python
import math
math.log(8)  # log of the vocabulary size = 2.0794415416798357
```
this is the loss we should expect if the model was predicting uniformly across the vocabulary.

In [17]:
'''
We are going to use model weight initialization to zero
'''
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, n_embed=10):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, n_embed)
        self.linear = nn.Linear(n_embed, vocab_size)

        nn.init.constant_(self.linear.weight, 0.0)
        nn.init.constant_(self.linear.bias, 0.0)
    def forward(self, x):
        embeds = self.embedding(x)  # (N, block_size, n_embed)
        logits = self.linear(embeds)
        return logits

In [18]:
m = TinyGPT(vocab_size)
logits = m(torch.tensor(X, dtype=torch.long))  # (N, block_size, vocab_size)
logits = logits.view(-1, vocab_size)
targets = torch.tensor(Y, dtype=torch.long).view(-1)
loss = loss_fn(logits, targets)
print(loss.item())  # Print the loss value

2.0794413089752197


/var/folders/j0/tr2332417vjbmj74tfc0dcd40000gn/T/ipykernel_80628/1286610971.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  targets = torch.tensor(Y, dtype=torch.long).view(-1)


> This is the loss we should expect if the model was predicting uniformly across the vocabulary.

## Model Training

In [30]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, n_embed=10):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, n_embed)
        self.linear = nn.Linear(n_embed, vocab_size)

    def forward(self, x):
        embeds = self.embedding(x)  # (N, block_size, n_embed)
        logits = self.linear(embeds)
        return logits
model = TinyGPT(vocab_size)

In [32]:
x = torch.tensor([encode('Hell')])
y = torch.tensor([encode('ello')])
x, y

(tensor([[1, 3, 4, 4]]), tensor([[3, 4, 4, 5]]))

In [33]:
logits = model(x)
B, T, C = logits.shape
logits = logits.view(B * T, C)  # Reshape to (B*T, C)
targets = y.view(B * T)  # Reshape to (B*T,)
loss = loss_fn(logits, targets)
print(loss.item())  # Print the loss value

1.9881606101989746


In [34]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
for step in range(100):
    logits = model(x)
    B, T, C = logits.shape
    logits = logits.view(B * T, C)
    targets = y.view(B * T)
    loss = loss_fn(logits, targets)

    optimizer.zero_grad()  # Clear previous gradients
    loss.backward()
    optimizer.step()
    if step % 10 == 0:
        print(f'Step {step}, Loss: {loss.item()}')

Step 0, Loss: 1.9881606101989746
Step 10, Loss: 1.237666130065918
Step 20, Loss: 0.7913758754730225
Step 30, Loss: 0.5536472797393799
Step 40, Loss: 0.44401341676712036
Step 50, Loss: 0.3968998193740845
Step 60, Loss: 0.37660378217697144
Step 70, Loss: 0.36721375584602356
Step 80, Loss: 0.3621010184288025
Step 90, Loss: 0.3589394688606262


> now sampling from the model

In [41]:
context = torch.tensor([[stoi['H']]])
for _ in range(4):
    logits = model(context)
    logits = logits[:, -1, :]  # Get the last time step's logits
    probs = F.softmax(logits, dim=-1)  # Convert logits to probabilities
    idx = torch.multinomial(probs, num_samples=1)  # Sample from the distribution
    print(itos[idx.item()], end='')  # Print the sampled character
    context = torch.cat((context, idx), dim=1)  # Append the sampled index to the context

elod